# 🚀 CircuitKit Quickstart — The Pipeline in 10 Minutes

This notebook walks through the complete **Discover → Evaluate → Prune → Export → Visualize** workflow using CircuitKit's `Pipeline` class.

- **Model:** GPT-2 (small, CPU-friendly)
- **Task:** IOI (Indirect Object Identification)
- **Algorithm:** EAP-IG (Edge Attribution Patching with Integrated Gradients)
- **Runtime:** ~5 minutes on CPU

---

## Setup

In [ ]:
# For Colab: uncomment and run the setup cells from 00_colab_setup.ipynb
# For local: pip install circuitkit

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import circuitkit as ck
from circuitkit import Pipeline

print(f"CircuitKit v{ck.__version__}")

## Step 1: Create a Pipeline

A `Pipeline` binds a model and task together, carrying state across each step.

- `"gpt2"` — loads the model via TransformerLens with discovery hook-points pre-enabled
- `task="ioi"` — Indirect Object Identification, a standard circuit discovery benchmark

In [ ]:
pipe = Pipeline("gpt2", task="ioi", precision="float32", output_dir="./results/quickstart")
print(pipe)

## Step 2: Discover a Circuit

EAP-IG attributes importance scores to every node (attention head or MLP layer) by computing integrated gradients over the edge attribution patching objective.

The result is a `Circuit` object containing the pruning artifact and per-node importance scores.

In [ ]:
pipe.discover(
    algorithm="eap-ig",
    level="node",
    sparsity=0.3,
    n_examples=32,
    batch_size=4,
    ig_steps=5,
)

print(pipe._circuit)

In [ ]:
# Top 10 most important nodes by score
pipe._circuit.top_nodes(10)

### Reading node names

- `A0.1` → Attention head 1 in layer 0
- `MLP 3` → MLP sublayer in layer 3

Higher scores = more important for the model's IOI behavior.

## Step 3: Evaluate the Circuit

Faithfulness evaluation answers: *does this circuit actually explain the behavior?*

- **Patching** (Pillar 1): Ablate circuit nodes → how much does performance drop?
- **Ablation** (Pillar 2): Keep only circuit nodes → how much performance is retained?

In [ ]:
pipe.evaluate(pillars=["patching", "ablation"], n_examples=32)

# The evaluation report is stored in pipe._eval_report
if pipe._eval_report:
    import json
    print(json.dumps(pipe._eval_report, indent=2, default=str))

## Step 4: Visualize the Circuit

Generate an interactive graph of the discovered circuit. Nodes are colored by importance score.

In [ ]:
# Save as HTML file
pipe.visualize(mode="graph", output="./results/quickstart/circuit_graph.html")
print("Graph saved to ./results/quickstart/circuit_graph.html")

In [ ]:
# ── Inline circuit graph in the notebook ──────────────────────────────────
# Calling visualize() without an output path returns a Plotly FigureWidget
# with an interactive threshold slider.  Scroll to zoom, drag to pan.
widget = pipe.visualize(mode="graph")
widget

## Step 5: Prune the Model

Structural pruning removes the least-important nodes identified by the circuit. This reduces model size while (ideally) preserving the discovered behavior.

In [ ]:
pipe.prune(sparsity=0.3, scope="both")
print("Pruning complete.")

## Step 6: Export the Pruned Checkpoint

Write the pruned model as a standard HuggingFace checkpoint for downstream use.

In [ ]:
checkpoint_path = pipe.export("./results/quickstart/pruned_checkpoint")
print(f"Checkpoint saved to: {checkpoint_path}")

## Step 7: Pipeline Summary

Review the full pipeline state — every step tracked.

In [ ]:
pipe.summary()

---
## Next Steps

- **02** — Compare 6 discovery algorithms on a real model
- **03** — Bring your own CSV data (jailbreak safety example)
- **04** — Deep dive into all 6 evaluation pillars
- **05** — Full visualization gallery
- **06** — Pruning, quantization, and selective finetuning
- **07** — CLI and YAML-driven workflows
- **08** — Advanced research tools (MasterGrid, IF, selectors)